# CerviGrade: attention head, settings sweep and 5-fold cross-validation
Same recipe as the earlier retrain (SIPaKMeD starting checkpoints, class weights 0.5, early stopping on validation macro F1, slide-level split with the same 1,723 test cells). The defaults in the settings cell reproduce that recipe exactly, so a plain run gives the earlier numbers again. Every setting you change is a new run in a new folder. The test set is never used to choose anything.

**What it can test**
- **Attention head.** `RT_HEAD = "attention"` keeps EfficientNetB0 and adds one small self-attention block on its 7 x 7 feature map (49 patches). The patches compare notes with each other before the final decision. The head keeps the ordinary pooled features too, so it can only add to the baseline, never replace it.
- **Learning rate, dropout and augmentation.** `RT_LR`, `RT_DROPOUT` and `RT_AUG` ("flips" or "strong": flips, 90-degree turns and small colour changes). Section 13 tries them one at a time and ranks them by validation macro F1.
- **5-fold cross-validation.** Section 14 trains five times, each time testing on a different fifth of the slides, and reports the average ± spread. Folds are made of whole slides, so no slide is ever in both training and testing.
- **Everything from before is still here:** larger crop (`CROP_SIZE`), the Negative cap, the leakage check, ConvNeXt and RegNet, and averaging (section 12).

**Inputs to add:** the CRIC dataset that holds the full slides (`classifications.csv` and the `images` folder, the same one the detector notebook uses), the Stage 1 `Training notebook` output, and your earlier retrain results Dataset (only for the comparison in section 11). GPU on. No Internet needed.

**Suggested order:**
1. Run the settings cell with `QUICK_TEST = True` once, to check everything runs, then set it back to `False`.
2. Run sections 2 to 5 (EfficientNetB0 baseline, three seeds). You should get the earlier numbers.
3. Run section 13 (the sweep). Pick the best setting by validation macro F1.
4. Put the winner in the settings cell, run section 5 again for three seeds, and compare in section 11.
5. Run section 14 once, with the final settings.

Seed noise is about ± 0.03 macro F1. A difference smaller than that is noise, not a finding.

## 1. Settings
The only cell you edit. Set `QUICK_TEST = True` first to check everything runs (a few minutes), then set it back to `False`. Change one setting per run, so you know what caused any difference.

In [ ]:
# ----- Retrain settings (the only cell you edit) -----
SEEDS = [1, 2, 3]            # one full training run per seed, for each model
RT_EPOCHS = 30               # upper limit; early stopping usually ends sooner
RT_PATIENCE = 5              # stop after this many epochs without a better validation macro F1
RT_LR = 1e-4
RT_DROPOUT = 0.3
CLASS_WEIGHT_POWER = 0.5     # 0 = off, 1 = full balancing. 0.5 = the earlier retrain (the recipe to beat).
RT_IMG = 224
RT_BATCH = 32
RT_WORKERS = 0              # 0 = load images in the main process. Safer after TensorFlow has started (extra worker processes can freeze RegNet)
QUICK_TEST = False           # True = one seed and one epoch per model, to check that everything runs
NEGATIVE_PER_EPOCH = None    # None = every Negative cell every epoch (the earlier retrain). A number, e.g. 1000, = that many randomly chosen Negative cells per epoch.
CROP_SIZE = 200              # Side, in original slide pixels, of the square cut around each expert dot. 200 = the earlier retrain. Larger = more surrounding context. Every crop is then resized to RT_IMG.
SPLIT_MODE = "slide"         # "slide" = every slide sits in one split only (the honest test, same 1,723 test cells). "random" = cells shuffled across splits (leaky, only to measure how much leakage would inflate the scores).
RT_HEAD = "linear"           # "linear" = pooled features and one dense layer (the earlier retrain). "attention" = adds a small self-attention block on the 7 x 7 feature map (EfficientNetB0 only).
RT_AUG = "flips"             # "flips" = random flips only (the earlier retrain). "strong" = flips, random 90-degree turns and small brightness, contrast, saturation and hue changes.
SAVE_MODELS = False          # True = also save the three chosen models as .keras / .pt files (slower, large files). The main notebook only needs the probabilities.

if QUICK_TEST:
    SEEDS, RT_EPOCHS, RT_PATIENCE = [1], 1, 1
    print("QUICK TEST: one seed and one epoch per model. The scores mean nothing.")
RT_ID = (f"w{CLASS_WEIGHT_POWER}" + (f"_n{NEGATIVE_PER_EPOCH}" if NEGATIVE_PER_EPOCH else "") + (f"_c{CROP_SIZE}" if CROP_SIZE != 200 else "")
         + ("_random" if SPLIT_MODE == "random" else "") + (f"_lr{RT_LR:g}" if RT_LR != 1e-4 else "") + (f"_do{RT_DROPOUT:g}" if RT_DROPOUT != 0.3 else "")
         + ("_attn" if RT_HEAD == "attention" else "") + ("_strongaug" if RT_AUG == "strong" else "") + f"_e{RT_EPOCHS}_p{RT_PATIENCE}")
RT_DIR = f"/kaggle/working/runs_{RT_ID}"
print("Finished runs are saved in", RT_DIR)
assert SPLIT_MODE in ("slide", "random"), "SPLIT_MODE must be \"slide\" or \"random\""
print(f"Crop size {CROP_SIZE} px, resized to {RT_IMG}. Split mode: {SPLIT_MODE}.")
assert RT_HEAD in ("linear", "attention"), "RT_HEAD must be \"linear\" or \"attention\""
assert RT_AUG in ("flips", "strong"), "RT_AUG must be \"flips\" or \"strong\""
print(f"Head: {RT_HEAD}. Augmentation: {RT_AUG}. Learning rate {RT_LR:g}, dropout {RT_DROPOUT:g}.")

## 2. Cut the crops and build the split
Crops are cut from the full slides around the experts' dots. The split is by source slide, seed 42, 70 / 15 / 15, the same slides as before, so the test count should be 1,723.

In [ ]:
import os, re, random
import pandas as pd

def find_containing(root, predicate, max_matches=5):
    matches = []
    for dirpath, dirnames, filenames in os.walk(root):
        if predicate(dirpath, dirnames, filenames):
            matches.append(dirpath)
            if len(matches) >= max_matches:
                break
    return matches

INPUT_ROOT = "/kaggle/input"
ckpt_files = {"efficientnetb0": "efficientnetb0_sipakmed_best.keras",
              "convnext": "convnext_sipakmed_best.keras",
              "regnet": "regnet_sipakmed_best.pt"}
checkpoints = {}
for key, fname in ckpt_files.items():
    hits = find_containing(INPUT_ROOT, lambda d, dn, fn, f=fname: f in fn, max_matches=1)
    checkpoints[key] = os.path.join(hits[0], fname) if hits else None
    print(f"  {key}: {'found' if hits else 'MISSING'} -> {checkpoints[key]}")
assert all(checkpoints.values()), "Add the Stage 1 'training-notebook' output as an input."

RANDOM_SEED = 42

# ----- Cut the crops from the full slides (the earlier runs used a fixed 200 px cut) -----
from PIL import Image
csv_path = next((os.path.join(d, "classifications.csv") for d, _, f in os.walk(INPUT_ROOT) if "classifications.csv" in f), None)
assert csv_path, "classifications.csv not found. Add the CRIC dataset that holds the full slides (the same one the detector notebook uses)."
CRIC_ROOT = os.path.dirname(csv_path)
CRIC_IMAGES_DIR = os.path.join(CRIC_ROOT, "images")
assert os.path.isdir(CRIC_IMAGES_DIR), f"No images folder next to {csv_path}"
cric_cells = pd.read_csv(csv_path)
cric_cells["on_disk"] = cric_cells["image_doi"].apply(lambda d: d.rstrip("/").split("figshare.")[-1]) + "_" + cric_cells["image_filename"]
CRIC_CROPPED_DIR = f"/kaggle/working/CRIC_crop{CROP_SIZE}"
half = CROP_SIZE // 2

def crop_path(row):
    return os.path.join(CRIC_CROPPED_DIR, row["bethesda_system"], f"{row['image_id']}_{row['cell_id']}.png")

n_new = 0
for on_disk, rows in cric_cells.groupby("on_disk"):
    todo = [r for _, r in rows.iterrows() if not os.path.exists(crop_path(r))]
    if not todo:
        continue
    img = Image.open(os.path.join(CRIC_IMAGES_DIR, on_disk)).convert("RGB")
    w, h = img.size
    assert w >= CROP_SIZE and h >= CROP_SIZE, f"CROP_SIZE {CROP_SIZE} is larger than the slide {on_disk} ({w} x {h})"
    for r in todo:
        cx, cy = int(r["nucleus_x"]), int(r["nucleus_y"])
        left = max(0, min(cx - half, w - CROP_SIZE))
        top = max(0, min(cy - half, h - CROP_SIZE))
        os.makedirs(os.path.dirname(crop_path(r)), exist_ok=True)
        tmp = crop_path(r) + ".tmp"          # write under a temporary name, so a stopped session never leaves a half-written crop behind
        img.crop((left, top, left + CROP_SIZE, top + CROP_SIZE)).save(tmp, format="PNG")
        os.replace(tmp, crop_path(r))
        n_new += 1
print(f"Crops of {CROP_SIZE} x {CROP_SIZE} px: {len(cric_cells)} cells, {n_new} newly cut, saved in {CRIC_CROPPED_DIR}")

def build_cric_manifest():
    return pd.DataFrame({"filepath": [crop_path(r) for _, r in cric_cells.iterrows()],
                         "label": cric_cells["bethesda_system"].values,
                         "dataset": "CRIC",
                         "group": ("CRIC_slide_" + cric_cells["image_id"].astype(str)).values})

def split_by_group_global(manifest_df, seed=RANDOM_SEED, train_frac=0.7, val_frac=0.15):
    rng = random.Random(seed)
    groups = sorted(manifest_df["group"].unique())
    rng.shuffle(groups)
    n = len(groups)
    n_train = round(n * train_frac)
    n_val = round(n * val_frac)
    group_split = {}
    for g in groups[:n_train]:
        group_split[g] = "train"
    for g in groups[n_train:n_train + n_val]:
        group_split[g] = "val"
    for g in groups[n_train + n_val:]:
        group_split[g] = "test"
    return manifest_df["group"].map(group_split)

def split_cells_randomly(manifest_df, seed=RANDOM_SEED, train_frac=0.7, val_frac=0.15):
    """Leaky split: cells of one slide land in train AND test. Only used to measure what leakage would add."""
    rng = np.random.RandomState(seed)
    out = np.empty(len(manifest_df), dtype=object)
    for label, idx in manifest_df.groupby("label").indices.items():
        idx = rng.permutation(idx)
        n_train, n_val = round(len(idx) * train_frac), round(len(idx) * val_frac)
        out[idx[:n_train]] = "train"
        out[idx[n_train:n_train + n_val]] = "val"
        out[idx[n_train + n_val:]] = "test"
    return pd.Series(out, index=manifest_df.index)

cric_manifest = build_cric_manifest()
if SPLIT_MODE == "random":
    cric_manifest["split"] = split_cells_randomly(cric_manifest)
    print("RANDOM SPLIT: slides are shared between train and test. These scores are inflated on purpose, never report them as the result.")
else:
    cric_manifest["split"] = split_by_group_global(cric_manifest)
print(cric_manifest["split"].value_counts().to_dict())

## 3. Class weights, scoring and helpers
Rare grades get a larger weight in the loss, computed from the training split only. This cell also holds the small helpers the training cells use.

In [ ]:
import os, time, json, copy, gc, random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, accuracy_score, f1_score

missing = [n for n in ("CRIC_CROPPED_DIR", "checkpoints", "cric_manifest") if n not in globals()]
assert not missing, f"Run the cells above first. Missing: {missing}"
os.makedirs(RT_DIR, exist_ok=True)

CKPT = {"EfficientNetB0": checkpoints["efficientnetb0"], "ConvNeXt": checkpoints["convnext"], "RegNet": checkpoints["regnet"]}
for arch, path in CKPT.items():
    assert os.path.isfile(path), f"Missing Stage 1 checkpoint for {arch}: {path}"

cric_classes = sorted(cric_manifest["label"].unique())
cric_class_to_idx = {c: i for i, c in enumerate(cric_classes)}
SHORT_NAMES = [c.replace("Negative for intraepithelial lesion", "Negative") for c in cric_classes]
RT_K = len(cric_classes)
cric_train_df = cric_manifest[cric_manifest["split"] == "train"]
cric_val_df = cric_manifest[cric_manifest["split"] == "val"]
cric_test_df = cric_manifest[cric_manifest["split"] == "test"]
rt_y_train = cric_train_df["label"].map(cric_class_to_idx).values
rt_y_val = cric_val_df["label"].map(cric_class_to_idx).values
y_true = cric_test_df["label"].map(cric_class_to_idx).values
rt_test_keys = (cric_test_df["label"] + "/" + cric_test_df["filepath"].map(os.path.basename)).values
print(f"CRIC  train {len(cric_train_df)}  val {len(cric_val_df)}  test {len(cric_test_df)}   (slide split: test should be 1,723)")

# Rare grades get a larger weight in the loss. The weights come from the training split only.
NEG_IDX = next(i for i, c in enumerate(cric_classes) if c.startswith("Negative"))
rt_counts = np.bincount(rt_y_train, minlength=RT_K)
if NEGATIVE_PER_EPOCH:
    rt_counts[NEG_IDX] = min(rt_counts[NEG_IDX], NEGATIVE_PER_EPOCH)       # cells the model really sees in one epoch
rt_w = (rt_counts.sum() / (RT_K * rt_counts)) ** CLASS_WEIGHT_POWER
rt_w = rt_w * rt_counts.sum() / (rt_w * rt_counts).sum()          # keep the average weight per image at 1
RT_CLASS_WEIGHTS = {i: float(rt_w[i]) for i in range(RT_K)}
print(pd.DataFrame({"train cells per epoch": rt_counts, "weight": np.round(rt_w, 2)}, index=SHORT_NAMES))

def rt_test_metrics(y_true, y_prob):
    y_pred = np.argmax(y_prob, axis=1)
    rep = classification_report(y_true, y_pred, target_names=SHORT_NAMES, output_dict=True, zero_division=0)
    out = {"accuracy": round(accuracy_score(y_true, y_pred), 3),
           "weighted_f1": round(rep["weighted avg"]["f1-score"], 3),
           "macro_f1": round(rep["macro avg"]["f1-score"], 3)}
    for c in SHORT_NAMES:
        out[f"recall_{c}"] = round(rep[c]["recall"], 3)
    return out

# ----- Bring back earlier results (only matters in a fresh session) -----
# A fresh session starts with an empty /kaggle/working. If you attached a Kaggle Dataset that holds the downloaded runs_* folder,
# its files are copied back here, so finished seeds are skipped instead of trained again.
import shutil
if not os.path.exists(os.path.join(RT_DIR, "runs.json")):
    found = [d for d, _, fns in os.walk(INPUT_ROOT) if "runs.json" in fns]
    # Only restore runs made with the same settings: a folder with the same name, or (with no Negative cap) a flat download of the earlier runs.
    candidates = [d for d in found if os.path.basename(d) == os.path.basename(RT_DIR)]
    if not candidates and not NEGATIVE_PER_EPOCH and CROP_SIZE == 200 and SPLIT_MODE == "slide" \
            and RT_HEAD == "linear" and RT_AUG == "flips" and RT_LR == 1e-4 and RT_DROPOUT == 0.3:
        candidates = [d for d in found if not os.path.basename(d).startswith("runs_")]
    if candidates:
        for fn in os.listdir(candidates[0]):
            if os.path.isfile(os.path.join(candidates[0], fn)):
                shutil.copy2(os.path.join(candidates[0], fn), os.path.join(RT_DIR, fn))
        print("Restored earlier runs from", candidates[0])
    else:
        print("No earlier runs with these settings found in the inputs: everything will train from scratch.")

RT_RUNS_FILE = os.path.join(RT_DIR, "runs.json")
rt_runs = json.load(open(RT_RUNS_FILE)) if os.path.exists(RT_RUNS_FILE) else {}
def rt_probs_path(arch, seed): return os.path.join(RT_DIR, f"probs_{arch}_seed{seed}.npz")
def rt_weights_path(arch, seed): return os.path.join(RT_DIR, f"weights_{arch}_seed{seed}" + (".pt" if arch == "RegNet" else ".weights.h5"))
def rt_run_done(arch, seed): return f"{arch}_seed{seed}" in rt_runs and os.path.exists(rt_probs_path(arch, seed))
print(f"{len(rt_runs)} finished run(s) found on disk.")

# ----- Helpers used by the three training cells -----
from IPython.display import display
RT_METRIC_COLS = ["accuracy", "weighted_f1", "macro_f1"] + [f"recall_{c}" for c in SHORT_NAMES]

def rt_mean_std(series):
    m, s = series.mean(), series.std(ddof=1)
    return f"{m:.3f} ± {s:.3f}" if len(series) > 1 else f"{m:.3f} (1 run)"

def rt_train_all(arch):
    """Train one model once per seed. Finished runs are skipped, so it is safe to run again."""
    durations = []
    for n, seed in enumerate(SEEDS, 1):
        key = f"{arch}_seed{seed}"
        if rt_run_done(arch, seed):
            print(f"[{n}/{len(SEEDS)}] {key}: already finished, skipping")
            continue
        t0 = time.time()
        info, probs = rt_run_regnet(seed) if arch == "RegNet" else rt_run_keras(arch, seed)
        minutes = (time.time() - t0) / 60
        durations.append(minutes)
        np.savez(rt_probs_path(arch, seed), y_prob=probs, y_true=y_true, test_keys=rt_test_keys)
        rt_runs[key] = {"architecture": arch, "seed": seed, "minutes": round(minutes, 1), **info, **rt_test_metrics(y_true, probs)}
        json.dump(rt_runs, open(RT_RUNS_FILE, "w"), indent=2)
        r = rt_runs[key]
        left = sum(1 for s in SEEDS[n:] if not rt_run_done(arch, s))
        print(f"[{n}/{len(SEEDS)}] {key}: {minutes:.1f} min | best epoch {r['best_epoch']} of {r['epochs_run']} | "
              f"val macro F1 {r['val_macro_f1']} | test acc {r['accuracy']} weighted F1 {r['weighted_f1']} "
              f"macro F1 {r['macro_f1']} SCC recall {r['recall_SCC']} | about {left * np.mean(durations):.0f} min left")
    rt_show(arch)

def rt_show(arch):
    sub = pd.DataFrame([r for r in rt_runs.values() if r["architecture"] == arch]).sort_values("seed")
    if QUICK_TEST:
        print("QUICK TEST: these scores come from one epoch and mean nothing.")
    print(f"\n{arch}: every seed")
    display(sub[["seed", "best_epoch", "epochs_run", "val_macro_f1"] + RT_METRIC_COLS].reset_index(drop=True))
    print(f"{arch}: average ± standard deviation across seeds")
    display(pd.DataFrame({c: [rt_mean_std(sub[c])] for c in RT_METRIC_COLS}, index=[f"{arch} ({len(sub)} runs)"]))

## 4. Keras training code (EfficientNetB0 and ConvNeXt)
Definitions only. Nothing trains here. Includes the augmentation options and the attention head.

In [ ]:
import tensorflow as tf
try:
    for gpu in tf.config.list_physical_devices("GPU"):
        tf.config.set_logical_device_configuration(gpu, [tf.config.LogicalDeviceConfiguration(memory_limit=10240)])
except Exception as e:
    print("GPU memory limit left as it was:", e)    # harmless; TensorFlow was already started by an earlier cell
from tensorflow.keras import layers, models
from tensorflow.keras.applications.efficientnet import preprocess_input as rt_effnet_pre
from tensorflow.keras.applications.convnext import preprocess_input as rt_convnext_pre
RT_PREPROCESS = {"EfficientNetB0": rt_effnet_pre, "ConvNeXt": rt_convnext_pre}

def rt_augment(image):
    """Pixels are still 0..255 here. Cells have no 'up', so flips and 90-degree turns never change the label."""
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    if RT_AUG == "strong":
        image = tf.image.rot90(image, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
        image = tf.image.random_brightness(image, 20.0)
        image = tf.image.random_contrast(image, 0.85, 1.15)
        image = tf.image.random_saturation(image, 0.85, 1.15)
        image = tf.image.random_hue(image, 0.02)
        image = tf.clip_by_value(image, 0.0, 255.0)
    return image

def rt_make_tf_dataset(df, preprocess_fn, shuffle_seed=None, augment=False):
    filepaths = df["filepath"].values
    labels = df["label"].map(cric_class_to_idx).values
    def load_and_preprocess(filepath, label):
        image = tf.io.read_file(filepath)
        image = tf.image.decode_image(image, channels=3, expand_animations=False)
        image = tf.image.resize(image, [RT_IMG, RT_IMG])
        if augment:
            image = rt_augment(image)
        return preprocess_fn(image), label
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle_seed is not None:
        ds = ds.shuffle(buffer_size=len(df), seed=shuffle_seed)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(RT_BATCH).prefetch(tf.data.AUTOTUNE)

@tf.keras.utils.register_keras_serializable(package="rt")
class RtPosEmbed(layers.Layer):
    """Adds a learned 'where am I' number to each of the 49 patches."""
    def build(self, input_shape):
        self.pos = self.add_weight(name="pos", shape=(1, input_shape[1], input_shape[2]),
                                   initializer=tf.keras.initializers.RandomNormal(stddev=0.02), trainable=True)
    def call(self, x):
        return x + self.pos

RT_ATTN_DIM, RT_ATTN_HEADS = 256, 4

def rt_feature_map_model(base):
    """The EfficientNetB0 base up to its last 4-D feature map (7 x 7 x 1280 for a 224 px input), before pooling."""
    layer = next(l for l in reversed(base.layers) if len(l.output.shape) == 4)
    return models.Model(base.input, layer.output)

def rt_attention_features(feat):
    """One transformer encoder block on the patches of a feature map. Returns the pooled ordinary features and the pooled attention features."""
    d = RT_ATTN_DIM
    plain = layers.GlobalAveragePooling2D()(feat)
    x = layers.Reshape((feat.shape[1] * feat.shape[2], feat.shape[3]))(feat)      # 49 patches, one row each
    x = layers.Dense(d)(x)
    x = RtPosEmbed()(x)
    h = layers.LayerNormalization(epsilon=1e-6)(x)
    h = layers.MultiHeadAttention(num_heads=RT_ATTN_HEADS, key_dim=d // RT_ATTN_HEADS, dropout=0.1)(h, h)   # every patch looks at every other patch
    x = layers.Add()([x, h])
    h = layers.LayerNormalization(epsilon=1e-6)(x)
    h = layers.Dense(2 * d, activation="gelu")(h)
    h = layers.Dropout(0.1)(h)
    h = layers.Dense(d)(h)
    x = layers.Add()([x, h])
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    return plain, layers.GlobalAveragePooling1D()(x)

def rt_build_keras(arch):
    loaded = tf.keras.models.load_model(CKPT[arch])
    base = next(l for l in loaded.layers if isinstance(l, tf.keras.Model))
    base.trainable = True
    inputs = layers.Input(shape=(RT_IMG, RT_IMG, 3))
    if RT_HEAD == "attention":
        assert arch == "EfficientNetB0", "RT_HEAD = 'attention' is built for EfficientNetB0 only"
        plain, attn = rt_attention_features(rt_feature_map_model(base)(inputs))
        x = layers.Concatenate()([plain, attn])           # the ordinary features stay, attention is added next to them
    else:
        x = base(inputs)
    x = layers.Dropout(RT_DROPOUT)(x)
    outputs = layers.Dense(RT_K, activation="softmax")(x)
    model = models.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=RT_LR),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def rt_make_train_dataset(preprocess_fn, seed):
    """Training data. With NEGATIVE_PER_EPOCH set, every epoch uses a fresh random subset of the Negative cells."""
    if not NEGATIVE_PER_EPOCH:
        return rt_make_tf_dataset(cric_train_df, preprocess_fn, shuffle_seed=seed, augment=True)
    filepaths = cric_train_df["filepath"].values
    labels = cric_train_df["label"].map(cric_class_to_idx).values
    is_neg = labels == NEG_IDX
    def load_and_augment(filepath, label):
        image = tf.io.read_file(filepath)
        image = tf.image.decode_image(image, channels=3, expand_animations=False)
        image = tf.image.resize(image, [RT_IMG, RT_IMG])
        image = rt_augment(image)
        return preprocess_fn(image), label
    negatives = tf.data.Dataset.from_tensor_slices((filepaths[is_neg], labels[is_neg]))
    negatives = negatives.shuffle(int(is_neg.sum()), seed=seed, reshuffle_each_iteration=True).take(NEGATIVE_PER_EPOCH)
    others = tf.data.Dataset.from_tensor_slices((filepaths[~is_neg], labels[~is_neg]))
    ds = negatives.concatenate(others).shuffle(len(cric_train_df), seed=seed + 1, reshuffle_each_iteration=True)
    ds = ds.map(load_and_augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(RT_BATCH).prefetch(tf.data.AUTOTUNE)

class MacroF1Stopping(tf.keras.callbacks.Callback):
    def __init__(self, val_ds, y_val, patience):
        super().__init__()
        self.val_ds, self.y_val, self.patience = val_ds, y_val, patience
        self.best, self.best_epoch, self.best_weights, self.wait = -1.0, 0, None, 0
    def on_epoch_end(self, epoch, logs=None):
        probs = self.model.predict(self.val_ds, verbose=0)
        f1 = f1_score(self.y_val, probs.argmax(1), average="macro", zero_division=0)
        logs["val_macro_f1"] = f1
        if f1 > self.best + 1e-6:
            self.best, self.best_epoch, self.wait = f1, epoch + 1, 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True
    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)

class RtEpochBar(tf.keras.callbacks.Callback):
    def __init__(self, max_epochs, title):
        super().__init__()
        self.max_epochs, self.title = max_epochs, title
    def on_train_begin(self, logs=None):
        self.outer = tqdm(total=self.max_epochs, desc=self.title, unit="epoch")
    def on_epoch_begin(self, epoch, logs=None):
        self.inner = tqdm(total=self.params.get("steps"), desc=f"  epoch {epoch + 1}", unit="batch", leave=False)
    def on_train_batch_end(self, batch, logs=None):
        self.inner.update(1)
    def on_epoch_end(self, epoch, logs=None):
        self.inner.close()
        self.outer.update(1)
        self.outer.set_postfix(loss=f"{logs['loss']:.3f}", acc=f"{logs['accuracy']:.3f}", val_macro_f1=f"{logs['val_macro_f1']:.3f}")
    def on_train_end(self, logs=None):
        self.outer.close()

def rt_run_keras(arch, seed):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    pre = RT_PREPROCESS[arch]
    train_ds = rt_make_train_dataset(pre, seed)
    val_ds = rt_make_tf_dataset(cric_val_df, pre)
    test_ds = rt_make_tf_dataset(cric_test_df, pre)
    model = rt_build_keras(arch)
    stopper = MacroF1Stopping(val_ds, rt_y_val, RT_PATIENCE)
    history = model.fit(train_ds, epochs=RT_EPOCHS, class_weight=RT_CLASS_WEIGHTS, verbose=0,
                        callbacks=[stopper, RtEpochBar(RT_EPOCHS, f"{arch} seed {seed}")])
    probs = model.predict(test_ds, verbose=0)
    model.save_weights(rt_weights_path(arch, seed))
    info = {"best_epoch": stopper.best_epoch, "epochs_run": len(history.history["loss"]), "val_macro_f1": round(stopper.best, 3)}
    del model, train_ds, val_ds, test_ds
    tf.keras.backend.clear_session(); gc.collect()
    return info, probs

## 5. Train EfficientNetB0
One run per seed, then its table. Finished runs are saved, so running the cell again skips them.

In [ ]:
rt_train_all("EfficientNetB0")

## 6. Train ConvNeXt-Tiny

In [ ]:
rt_train_all("ConvNeXt")

## 7. RegNetY-400MF training code (PyTorch)
Definitions only. Nothing trains here.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import regnet_y_400mf
from PIL import Image

rt_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch device:", rt_device)

_mean, _std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
rt_train_tf = transforms.Compose([transforms.Resize((RT_IMG, RT_IMG)), transforms.RandomHorizontalFlip(),
                                  transforms.RandomVerticalFlip(), transforms.ToTensor(), transforms.Normalize(_mean, _std)])
rt_eval_tf = transforms.Compose([transforms.Resize((RT_IMG, RT_IMG)), transforms.ToTensor(), transforms.Normalize(_mean, _std)])

class RtCricDataset(Dataset):
    def __init__(self, df, transform):
        self.filepaths = df["filepath"].values
        self.labels = df["label"].map(cric_class_to_idx).values
        self.transform = transform
    def __len__(self):
        return len(self.filepaths)
    def __getitem__(self, idx):
        return self.transform(Image.open(self.filepaths[idx]).convert("RGB")), int(self.labels[idx])

def rt_build_regnet():
    model = regnet_y_400mf(weights=None)
    model.fc = torch.nn.Linear(model.fc.in_features, 5)          # the 5-class SIPaKMeD head, so the saved weights fit
    model.load_state_dict(torch.load(CKPT["RegNet"], map_location="cpu"))
    model.fc = torch.nn.Sequential(torch.nn.Dropout(RT_DROPOUT), torch.nn.Linear(model.fc.in_features, RT_K))
    for p in model.parameters():
        p.requires_grad = True
    return model.to(rt_device)

def rt_torch_probs(model, loader, desc):
    model.eval()
    out = []
    with torch.no_grad():
        for images, _ in tqdm(loader, desc=desc, unit="batch", leave=False):
            out.append(torch.softmax(model(images.to(rt_device)), dim=1).cpu().numpy())
    return np.concatenate(out)

class RtNegativeCapSampler(torch.utils.data.Sampler):
    """Every epoch: all non-Negative cells plus a fresh random subset of the Negative cells."""
    def __init__(self, labels, neg_idx, cap, seed):
        labels = np.asarray(labels)
        self.neg = np.where(labels == neg_idx)[0]
        self.rest = np.where(labels != neg_idx)[0]
        self.cap = min(cap, len(self.neg))
        self.rng = np.random.default_rng(seed)
    def __len__(self):
        return len(self.rest) + self.cap
    def __iter__(self):
        idx = np.concatenate([self.rest, self.rng.choice(self.neg, self.cap, replace=False)])
        self.rng.shuffle(idx)
        return iter(idx.tolist())

def rt_run_regnet(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    g = torch.Generator(); g.manual_seed(seed)
    train_set = RtCricDataset(cric_train_df, rt_train_tf)
    if NEGATIVE_PER_EPOCH:
        train_loader = DataLoader(train_set, batch_size=RT_BATCH, sampler=RtNegativeCapSampler(train_set.labels, NEG_IDX, NEGATIVE_PER_EPOCH, seed), num_workers=RT_WORKERS)
    else:
        train_loader = DataLoader(train_set, batch_size=RT_BATCH, shuffle=True, num_workers=RT_WORKERS, generator=g)
    val_loader = DataLoader(RtCricDataset(cric_val_df, rt_eval_tf), batch_size=RT_BATCH, shuffle=False, num_workers=RT_WORKERS)
    test_loader = DataLoader(RtCricDataset(cric_test_df, rt_eval_tf), batch_size=RT_BATCH, shuffle=False, num_workers=RT_WORKERS)
    model = rt_build_regnet()
    optimizer = torch.optim.Adam(model.parameters(), lr=RT_LR)
    criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor([RT_CLASS_WEIGHTS[i] for i in range(RT_K)], dtype=torch.float32, device=rt_device))
    best, best_epoch, best_state, wait, epochs_run = -1.0, 0, None, 0, 0
    bar = tqdm(range(RT_EPOCHS), desc=f"RegNet seed {seed}", unit="epoch")
    for epoch in bar:
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for images, labels in tqdm(train_loader, desc=f"  epoch {epoch + 1}", unit="batch", leave=False):
            images, labels = images.to(rt_device), labels.to(rt_device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)
        epochs_run += 1
        f1 = f1_score(rt_y_val, rt_torch_probs(model, val_loader, "  validating").argmax(1), average="macro", zero_division=0)
        bar.set_postfix(loss=f"{total_loss / total:.3f}", acc=f"{correct / total:.3f}", val_macro_f1=f"{f1:.3f}")
        if f1 > best + 1e-6:
            best, best_epoch, wait = f1, epoch + 1, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= RT_PATIENCE:
                break
    model.load_state_dict(best_state)
    probs = rt_torch_probs(model, test_loader, "testing")
    torch.save(model.state_dict(), rt_weights_path("RegNet", seed))
    info = {"best_epoch": best_epoch, "epochs_run": epochs_run, "val_macro_f1": round(best, 3)}
    del model, optimizer
    gc.collect(); torch.cuda.empty_cache()
    return info, probs

## 8. Train RegNetY-400MF

In [ ]:
rt_train_all("RegNet")

## 9. All three side by side
Every run, then the average ± standard deviation per model. Differences smaller than the spread between seeds are noise.

In [ ]:
# ----- All three models side by side -----
rt_order = {a: i for i, a in enumerate(["EfficientNetB0", "ConvNeXt", "RegNet"])}
rt_df = pd.DataFrame(rt_runs.values())
assert len(rt_df), "No finished runs yet. Run the three training cells above first."
rt_df = rt_df.assign(_o=rt_df["architecture"].map(rt_order)).sort_values(["_o", "seed"]).drop(columns="_o").reset_index(drop=True)
missing_models = [a for a in rt_order if a not in set(rt_df["architecture"])]
if missing_models:
    print("Not trained yet:", missing_models)
if QUICK_TEST:
    print("QUICK TEST: the scores below come from one epoch and mean nothing.")
print("Every run")
display(rt_df[["architecture", "seed", "best_epoch", "epochs_run", "val_macro_f1"] + RT_METRIC_COLS])

rt_summary = rt_df.groupby("architecture")[RT_METRIC_COLS].agg(rt_mean_std)
rt_summary = rt_summary.loc[[a for a in rt_order if a in rt_summary.index]]
rt_summary.insert(0, "runs", rt_df.groupby("architecture").size())
print("Average ± standard deviation across seeds")
display(rt_summary)
rt_summary.to_csv("/kaggle/working/retrain_summary.csv")
rt_df.to_csv("/kaggle/working/retrain_all_runs.csv", index=False)

## 10. Pick the seed for each model and save the results
The seed with the best **validation** macro F1 is kept for each model. Its stored test probabilities are written to `/kaggle/working`, which is all the main notebook reads. With `SAVE_MODELS = True` the models themselves are also rebuilt, saved, reloaded and checked.

In [ ]:
# ----- Pick one seed per model by validation score and save the results -----
rt_best_seed = {}
for arch in rt_order:
    sub = rt_df[rt_df["architecture"] == arch]
    if len(sub):
        rt_best_seed[arch] = int(sub.sort_values(["val_macro_f1", "seed"], ascending=[False, True]).iloc[0]["seed"])
print("Chosen seed per model (best validation macro F1):", rt_best_seed)

rt_bundle = {"y_true": y_true, "test_keys": rt_test_keys}
for arch, seed in rt_best_seed.items():
    rt_bundle[arch] = np.load(rt_probs_path(arch, seed))["y_prob"]
np.savez("/kaggle/working/stage2_retrain_probs.npz", **rt_bundle)
json.dump({"best_seed": rt_best_seed, "settings_id": RT_ID, "class_weight_power": CLASS_WEIGHT_POWER, "negative_per_epoch": NEGATIVE_PER_EPOCH, "crop_size": CROP_SIZE, "split_mode": SPLIT_MODE},
          open("/kaggle/working/retrain_selection.json", "w"), indent=2)
print("Saved stage2_retrain_probs.npz and retrain_selection.json")

if SAVE_MODELS:
    # Rebuild each chosen model, save it, reload it and check it against the stored test probabilities.
    for arch, seed in rt_best_seed.items():
        if arch == "RegNet":
            m = rt_build_regnet()
            m.load_state_dict(torch.load(rt_weights_path(arch, seed), map_location=rt_device))
            check = rt_torch_probs(m, DataLoader(RtCricDataset(cric_test_df, rt_eval_tf), batch_size=RT_BATCH, num_workers=RT_WORKERS), "re-checking")
            torch.save(m.state_dict(), "/kaggle/working/regnet_cric_retrained.pt")
            del m
        else:
            tf.keras.backend.clear_session()
            m = rt_build_keras(arch)
            m.load_weights(rt_weights_path(arch, seed))
            check = m.predict(rt_make_tf_dataset(cric_test_df, RT_PREPROCESS[arch]), verbose=0)
            m.save(f"/kaggle/working/{arch.lower()}_cric_retrained.keras")
            del m
        gap = np.abs(check - rt_bundle[arch]).max()
        print(f"{arch}: saved the seed {seed} model. Reloaded model matches the stored test probabilities (largest gap {gap:.4f}).")
        if gap > 0.02:
            print("  WARNING: the reloaded model does not match. Do not use this saved file.")
else:
    print("SAVE_MODELS is False: the model files were not saved. The weights stay in the runs folder.")

## 11. Compare with the earlier run
Same test cells, same metrics. Differences smaller than the spread between seeds are noise.

In [ ]:
# ----- Compare with the earlier run (class weights 0.5, every Negative cell every epoch) -----
import glob
base_files = glob.glob("/kaggle/input/**/retrain_all_runs.csv", recursive=True)
if not base_files:
    print("Attach the earlier retrain results Dataset to see the comparison.")
else:
    base = pd.read_csv(base_files[0])
    rows = {}
    for arch in [a for a in rt_order if a in set(rt_df["architecture"])]:
        earlier = base[base["architecture"] == arch]
        this_run = rt_df[rt_df["architecture"] == arch]
        if len(earlier):
            rows[f"{arch}: earlier (crop 200, weights 0.5, all Negative cells, slide split)"] = {c: rt_mean_std(earlier[c]) for c in RT_METRIC_COLS}
        rows[f"{arch}: this run (crop {CROP_SIZE}, Negative cap {NEGATIVE_PER_EPOCH}, weights {CLASS_WEIGHT_POWER}, {SPLIT_MODE} split)"] = {c: rt_mean_std(this_run[c]) for c in RT_METRIC_COLS}
    comparison = pd.DataFrame(rows).T
    display(comparison)
    comparison.to_csv("/kaggle/working/retrain_comparison.csv")
    if SPLIT_MODE == "random":
        print("RANDOM SPLIT: the test cells differ from the earlier run, so this row is NOT a fair comparison. It shows how much leakage inflates the scores.")
    print("Read it by recall: Negative recall shows the false alarms on healthy cells, SCC recall shows the cancer-grade catch rate, macro F1 shows the overall balance. Accuracy is dominated by the Negative class (59% of the test cells).")

## 12. Average the models (no retraining)
Uses the test probabilities already saved by each finished run. One row per model for the single runs, then the seeds of each model averaged into one prediction, then every finished run averaged into one. The last table is the confusion matrix of that average: rows are the true grade, columns the grade the model chose.

In [ ]:
from sklearn.metrics import confusion_matrix
done_runs = [(a, s) for a in rt_order for s in SEEDS if rt_run_done(a, s)]
assert done_runs, "No finished runs yet."
probs_all = {k: np.load(rt_probs_path(*k))["y_prob"] for k in done_runs}
runs_df = pd.DataFrame(rt_runs.values())
avg_rows = {}
for a in rt_order:
    sub = runs_df[runs_df["architecture"] == a]
    if len(sub):
        avg_rows[f"{a}: single runs, mean of {len(sub)}"] = {c: round(sub[c].mean(), 3) for c in RT_METRIC_COLS}
    got = [probs_all[(a, s)] for s in SEEDS if (a, s) in probs_all]
    if len(got) > 1:
        avg_rows[f"{a}: {len(got)} seeds averaged into one"] = rt_test_metrics(y_true, np.mean(got, axis=0))
avg_p = np.mean(list(probs_all.values()), axis=0)
if len(probs_all) > 1:
    avg_rows[f"all {len(probs_all)} runs averaged into one"] = rt_test_metrics(y_true, avg_p)
avg_table = pd.DataFrame(avg_rows).T
display(avg_table)
avg_table.to_csv("/kaggle/working/retrain_averaging.csv")
cm = confusion_matrix(y_true, avg_p.argmax(1), labels=range(RT_K))
print("Confusion matrix of the averaged prediction (rows = true grade, columns = chosen grade)")
display(pd.DataFrame(cm, index=[f"true {n}" for n in SHORT_NAMES], columns=[f"pred {n}" for n in SHORT_NAMES]))

## 13. Settings sweep (EfficientNetB0)
Tries one change at a time on top of the baseline: learning rate, dropout, stronger augmentation and the attention head. Each setting trains once per seed in `SWEEP_SEEDS`. Settings are ranked by **validation** macro F1, never by test. Finished runs are saved, so running the cell again skips them, and you can stop and continue later.

Read the table with care: seed noise is about ± 0.03 macro F1, so a setting only counts if it beats the baseline by clearly more than that. The two best are worth running again with three seeds (put them in the settings cell and use section 5). Time: about as long as one EfficientNetB0 run per row and seed.

In [ ]:
# ----- Settings sweep: one change at a time, ranked by validation macro F1 -----
SWEEP_SEEDS = [1, 2]
SWEEP = [("baseline", {}),
         ("learning rate 3e-5", {"RT_LR": 3e-5}),
         ("learning rate 3e-4", {"RT_LR": 3e-4}),
         ("dropout 0.2", {"RT_DROPOUT": 0.2}),
         ("dropout 0.5", {"RT_DROPOUT": 0.5}),
         ("strong augmentation", {"RT_AUG": "strong"}),
         ("attention head", {"RT_HEAD": "attention"})]
SWEEP_DIR = f"/kaggle/working/sweep_c{CROP_SIZE}_w{CLASS_WEIGHT_POWER}" + (f"_n{NEGATIVE_PER_EPOCH}" if NEGATIVE_PER_EPOCH else "")
SWEEP_FILE = os.path.join(SWEEP_DIR, "sweep_runs.json")
os.makedirs(SWEEP_DIR, exist_ok=True)
if not os.path.exists(SWEEP_FILE):
    prev = [d for d, _, fns in os.walk(INPUT_ROOT) if "sweep_runs.json" in fns and os.path.basename(d) == os.path.basename(SWEEP_DIR)]
    if prev:
        for fn in os.listdir(prev[0]):
            if os.path.isfile(os.path.join(prev[0], fn)):
                shutil.copy2(os.path.join(prev[0], fn), os.path.join(SWEEP_DIR, fn))
        print("Restored earlier sweep runs from", prev[0])
sweep_runs = json.load(open(SWEEP_FILE)) if os.path.exists(SWEEP_FILE) else {}

def rt_sweep():
    saved = {n: globals()[n] for n in ("RT_LR", "RT_DROPOUT", "RT_AUG", "RT_HEAD", "RT_DIR")}
    durations, total = [], len(SWEEP) * len(SWEEP_SEEDS)
    try:
        globals()["RT_DIR"] = SWEEP_DIR                      # the weights files of sweep runs must not overwrite the main runs
        for name, change in SWEEP:
            for seed in SWEEP_SEEDS:
                key = f"{name}|{seed}"
                done = len(sweep_runs)
                if key in sweep_runs:
                    continue
                globals().update(saved); globals().update(change); globals()["RT_DIR"] = SWEEP_DIR
                print(f"[{done + 1}/{total}] {name}, seed {seed}  (lr {RT_LR:g}, dropout {RT_DROPOUT:g}, aug {RT_AUG}, head {RT_HEAD})")
                t0 = time.time()
                info, probs = rt_run_keras("EfficientNetB0", seed)
                minutes = (time.time() - t0) / 60
                durations.append(minutes)
                np.savez(os.path.join(SWEEP_DIR, f"probs_{name.replace(' ', '_')}_seed{seed}.npz"), y_prob=probs, y_true=y_true, test_keys=rt_test_keys)
                sweep_runs[key] = {"setting": name, "seed": seed, "minutes": round(minutes, 1), **info, **rt_test_metrics(y_true, probs)}
                json.dump(sweep_runs, open(SWEEP_FILE, "w"), indent=2)
                left = total - len(sweep_runs)
                print(f"    {minutes:.1f} min | best epoch {info['best_epoch']} of {info['epochs_run']} | validation macro F1 {info['val_macro_f1']} | about {left * np.mean(durations):.0f} min left")
    finally:
        globals().update(saved)                              # always put the settings back

def rt_sweep_table():
    df = pd.DataFrame(sweep_runs.values())
    order = [n for n, _ in SWEEP]
    g = df.groupby("setting")
    out = pd.DataFrame({"runs": g.size(),
                        "validation macro F1": g["val_macro_f1"].agg(rt_mean_std),
                        "_mean": g["val_macro_f1"].mean(),
                        "best epoch (avg)": g["best_epoch"].mean().round(1),
                        "minutes per run": g["minutes"].mean().round(1)})
    base_mean = out.loc["baseline", "_mean"] if "baseline" in out.index else np.nan
    out["vs baseline"] = (out["_mean"] - base_mean).round(3)
    out = out.sort_values("_mean", ascending=False).drop(columns="_mean")
    display(out)
    print("Ranked by validation macro F1. A gap to the baseline smaller than about 0.03 is noise.")
    df.to_csv(os.path.join(SWEEP_DIR, "sweep_all_runs.csv"), index=False)
    return out

if QUICK_TEST:
    print("QUICK TEST is on: the sweep would run one epoch per setting. Set QUICK_TEST = False in the settings cell for a real sweep.")
rt_sweep()
sweep_table = rt_sweep_table()

## 14. Five-fold cross-validation (grouped by slide)
The test set so far is one fixed slice (only 34 SCC cells). Here the 400 slides are cut into five groups of whole slides. The model is trained five times, each time tested on a different group, so **every cell is tested exactly once** (161 SCC cells in total). Inside each round, a slice of the remaining slides is kept for early stopping, so the tested group is never used to choose anything.

Run it **once, with the final settings** from the settings cell. It takes about five times one run. Finished folds are saved, so you can stop and continue. The result is an average ± spread per fold, plus one pooled score over all out-of-fold predictions.

In [ ]:
# ----- 5-fold cross-validation, folds made of whole slides -----
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import confusion_matrix
KFOLD_K = 5
KFOLD_SEED = 1                 # training seed used in every fold
KFOLD_ARCH = "EfficientNetB0"  # or "ConvNeXt"
KF_DIR = f"/kaggle/working/kfold_{KFOLD_ARCH}_{RT_ID}"
KF_FILE = os.path.join(KF_DIR, "kfold_runs.json")
os.makedirs(KF_DIR, exist_ok=True)
if not os.path.exists(KF_FILE):
    prev = [d for d, _, fns in os.walk(INPUT_ROOT) if "kfold_runs.json" in fns and os.path.basename(d) == os.path.basename(KF_DIR)]
    if prev:
        for fn in os.listdir(prev[0]):
            if os.path.isfile(os.path.join(prev[0], fn)):
                shutil.copy2(os.path.join(prev[0], fn), os.path.join(KF_DIR, fn))
        print("Restored earlier folds from", prev[0])
kf_runs = json.load(open(KF_FILE)) if os.path.exists(KF_FILE) else {}

kf_all = cric_manifest.reset_index(drop=True)
kf_y = kf_all["label"].map(cric_class_to_idx).values
kf_folds = list(StratifiedGroupKFold(n_splits=KFOLD_K, shuffle=True, random_state=RANDOM_SEED).split(kf_all, kf_y, kf_all["group"]))

def rt_kfold():
    names = ("cric_train_df", "cric_val_df", "cric_test_df", "rt_y_val", "y_true", "RT_CLASS_WEIGHTS", "RT_DIR")
    saved = {n: globals()[n] for n in names}
    durations = []
    try:
        globals()["RT_DIR"] = KF_DIR
        for f, (tr_idx, te_idx) in enumerate(kf_folds, 1):
            if f"fold{f}" in kf_runs:
                print(f"fold {f}/{KFOLD_K}: already finished, skipping")
                continue
            tr_part = kf_all.iloc[tr_idx].reset_index(drop=True)
            y_part = tr_part["label"].map(cric_class_to_idx).values
            fit_idx, val_idx = next(StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=RANDOM_SEED).split(tr_part, y_part, tr_part["group"]))
            train_df, val_df = tr_part.iloc[fit_idx], tr_part.iloc[val_idx]
            test_df = kf_all.iloc[te_idx]
            assert not (set(train_df["group"]) & set(test_df["group"])) and not (set(val_df["group"]) & set(test_df["group"])), "slide leakage"
            counts = np.bincount(train_df["label"].map(cric_class_to_idx).values, minlength=RT_K)
            if NEGATIVE_PER_EPOCH:
                counts[NEG_IDX] = min(counts[NEG_IDX], NEGATIVE_PER_EPOCH)
            w = (counts.sum() / (RT_K * counts)) ** CLASS_WEIGHT_POWER
            w = w * counts.sum() / (w * counts).sum()
            globals().update({"cric_train_df": train_df, "cric_val_df": val_df, "cric_test_df": test_df,
                              "rt_y_val": val_df["label"].map(cric_class_to_idx).values,
                              "y_true": test_df["label"].map(cric_class_to_idx).values,
                              "RT_CLASS_WEIGHTS": {i: float(w[i]) for i in range(RT_K)}})
            test_keys = (test_df["label"] + "/" + test_df["filepath"].map(os.path.basename)).values
            print(f"fold {f}/{KFOLD_K}: train {len(train_df)} | validation {len(val_df)} | test {len(test_df)} cells from {test_df['group'].nunique()} slides | SCC in test: {int((globals()['y_true'] == cric_class_to_idx['SCC']).sum())}")
            t0 = time.time()
            info, probs = rt_run_keras(KFOLD_ARCH, KFOLD_SEED)
            minutes = (time.time() - t0) / 60
            durations.append(minutes)
            np.savez(os.path.join(KF_DIR, f"probs_fold{f}.npz"), y_prob=probs, y_true=globals()["y_true"], test_keys=test_keys)
            kf_runs[f"fold{f}"] = {"fold": f, "minutes": round(minutes, 1), "test_cells": len(test_df), **info, **rt_test_metrics(globals()["y_true"], probs)}
            json.dump(kf_runs, open(KF_FILE, "w"), indent=2)
            r = kf_runs[f"fold{f}"]
            print(f"    {minutes:.1f} min | best epoch {r['best_epoch']} of {r['epochs_run']} | test acc {r['accuracy']} macro F1 {r['macro_f1']} SCC recall {r['recall_SCC']} | about {(KFOLD_K - len(kf_runs)) * np.mean(durations):.0f} min left")
    finally:
        globals().update(saved)

if QUICK_TEST:
    print("QUICK TEST is on: every fold would train for one epoch. Set QUICK_TEST = False in the settings cell for a real run.")
rt_kfold()

# ----- Results: per fold, average ± spread, and one pooled score over all out-of-fold cells -----
assert len(kf_runs) == KFOLD_K, f"Only {len(kf_runs)} of {KFOLD_K} folds finished. Run the cell again to continue."
kf_df = pd.DataFrame(kf_runs.values()).sort_values("fold").reset_index(drop=True)
print(f"{KFOLD_ARCH}, {RT_ID}: every fold")
display(kf_df[["fold", "test_cells", "best_epoch", "epochs_run", "val_macro_f1"] + RT_METRIC_COLS])
print(f"Average ± standard deviation over {KFOLD_K} folds")
kf_summary = pd.DataFrame({c: [rt_mean_std(kf_df[c])] for c in RT_METRIC_COLS}, index=[f"{KFOLD_ARCH} ({KFOLD_K} folds)"])
display(kf_summary)

kf_probs = [np.load(os.path.join(KF_DIR, f"probs_fold{f}.npz"), allow_pickle=True) for f in range(1, KFOLD_K + 1)]
oof_prob = np.concatenate([p["y_prob"] for p in kf_probs])
oof_true = np.concatenate([p["y_true"] for p in kf_probs])
oof_keys = np.concatenate([p["test_keys"] for p in kf_probs])
assert len(set(oof_keys)) == len(oof_keys) == len(kf_all), "every cell must be tested exactly once"
pooled = rt_test_metrics(oof_true, oof_prob)
print(f"Pooled over all {len(oof_true)} cells (each tested once, {int((oof_true == cric_class_to_idx['SCC']).sum())} SCC cells):")
display(pd.DataFrame(pooled, index=["pooled"]))
oof_pred = oof_prob.argmax(1)
cm = confusion_matrix(oof_true, oof_pred, labels=range(RT_K))
print("Confusion matrix (rows = expert label, columns = model prediction), pooled")
display(pd.DataFrame(cm, index=[f"true {c}" for c in SHORT_NAMES], columns=[f"pred {c}" for c in SHORT_NAMES]))

# Three-tier view of the same predictions: Negative | low-grade (ASC-US + LSIL) | high-grade (ASC-H + HSIL + SCC)
TIER_MEMBERS = [["Negative"], ["ASC-US", "LSIL"], ["ASC-H", "HSIL", "SCC"]]
tier_cols = [[SHORT_NAMES.index(c) for c in members] for members in TIER_MEMBERS]
def rt_to_tiers(p): return np.stack([p[:, cols].sum(axis=1) for cols in tier_cols], axis=1)
tier_true = np.array([next(t for t, cols in enumerate(tier_cols) if y in cols) for y in oof_true])
tier_pred = rt_to_tiers(oof_prob).argmax(1)
print(f"Three-tier view, pooled: accuracy {accuracy_score(tier_true, tier_pred):.3f}, macro F1 {f1_score(tier_true, tier_pred, average='macro'):.3f}")
kf_df.to_csv(os.path.join(KF_DIR, "kfold_per_fold.csv"), index=False)
np.savez(os.path.join(KF_DIR, "kfold_out_of_fold.npz"), y_prob=oof_prob, y_true=oof_true, test_keys=oof_keys)
print("Saved kfold_per_fold.csv and kfold_out_of_fold.npz in", KF_DIR)